# Deteksi dan Pengenalan Pelat Nomor Kendaraan — Tahap 1: dataset dan preprocessing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t1/blob/main/notebooks/01_dataset_preprocessing.ipynb)

Kelompok `pcd2026-ti23ka-t1` · 4 anggota · minimal **200 citra** (termasuk 60 foto sendiri).

Notebook ini membaca dataset dari folder Google Drive kelompok, memeriksa anotasi, menerapkan preprocessing, dan menyimpan gambar sebelum–sesudah. Jalankan dari atas ke bawah.

**Status pengumpulan per 14 November 2026:** 60 foto sendiri sudah berada di Drive dan masih ada pekerjaan anotasi. Notebook ini membaca isi Drive saat dijalankan; jumlah hasil selalu mengikuti data sebenarnya.

In [2]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t1"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23ka-t1")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas', 'x', 'y', 'w', 'h', 'teks_pelat']
JUMLAH_ANGGOTA = 4
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Mounted at /content/drive
Citra   : /content/drive/MyDrive/pcd2026-ti23ka-t1/citra
Anotasi : /content/drive/MyDrive/pcd2026-ti23ka-t1/anotasi.csv (ada)


## 1. Statistik dataset

Menampilkan jumlah dan rentang ukuran seluruh citra aktif dari Google Drive. Jumlah citra fisik dan jumlah anotasi dapat berbeda selama proses validasi berlangsung.

In [3]:
berkas = util.daftar_citra(FOLDER_CITRA)
if not berkas:
    raise RuntimeError(f"Tidak ada citra di {FOLDER_CITRA}")
stat = util.statistik_dataset(berkas)
print(f"{stat['jumlah']} citra ({stat['terbaca']} terbaca); minimal 200 untuk kelompok ini")
print('Lebar :', stat.get('lebar'))
print('Tinggi:', stat.get('tinggi'))
if stat['gagal']:
    print('Gagal dibaca:', stat['gagal'][:10])

200 citra (200 terbaca); minimal 200 untuk kelompok ini
Lebar : {'min': 420, 'median': 860, 'maks': 4000}
Tinggi: {'min': 561, 'median': 1080, 'maks': 3000}


## 2. Cek anotasi

Kotak lokasi pelat (x, y, lebar, tinggi dalam piksel) dan teks pelat tanpa spasi.

Kolom `anotasi.csv` (persis): `berkas, x, y, w, h, teks_pelat`. Contoh baris ada di `data/format_anotasi.csv`.

In [4]:
KOLOM_BERKAS = ['berkas']
if BERKAS_ANOTASI.exists():
    anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
    cek = util.cek_anotasi(anotasi, FOLDER_CITRA, KOLOM_BERKAS)
    print(len(anotasi), "baris anotasi")
    print("Disebut di anotasi tetapi berkasnya tidak ada:", cek["hilang"][:20])
    print("Citra belum beranotasi:", len(cek["tanpa_anotasi"]), cek["tanpa_anotasi"][:10])
else:
    print("anotasi.csv belum ada di", BERKAS_ANOTASI)
# Validasi tambahan batas koordinat untuk semua anotasi aktif.
if BERKAS_ANOTASI.exists():
    rusak = []
    for r in anotasi:
        p = FOLDER_CITRA / r['berkas']
        if not p.is_file():
            continue
        im = util.baca_citra(p)
        if im is None:
            rusak.append((r['berkas'], 'gagal dibaca'))
            continue
        H, W = im.shape[:2]
        try:
            x, y, w, h = (int(float(r[k])) for k in ('x','y','w','h'))
            if not (0 <= x < W and 0 <= y < H and 0 < w <= W-x and 0 < h <= H-y):
                rusak.append((r['berkas'], 'bbox di luar gambar'))
        except (ValueError, TypeError):
            rusak.append((r['berkas'], 'bbox bukan angka'))
    print('Anotasi dengan koordinat bermasalah:', len(rusak))
    for item in rusak[:10]: print('  ',item)

200 baris anotasi
Disebut di anotasi tetapi berkasnya tidak ada: []
Citra belum beranotasi: 0 []
Anotasi dengan koordinat bermasalah: 0


## 3. Preprocessing

Tahapan resize, grayscale, CLAHE, dan unsharp masking digunakan dengan parameter tetap untuk semua citra.

In [5]:

def praproses(citra):
    """
    Preprocessing citra pelat nomor kendaraan.

    Tahapan:
    1. Resize maksimal 1024 piksel
    2. Grayscale
    3. Peningkatan kontras CLAHE
    4. Penajaman citra
    """

    # 1. Resize dengan mempertahankan rasio gambar
    hasil = util.ubah_ukuran_maks(citra, 1024)

    # 2. Konversi BGR menjadi grayscale
    abu = cv2.cvtColor(hasil, cv2.COLOR_BGR2GRAY)

    # 3. Peningkatan kontras dengan CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    kontras = clahe.apply(abu)

    # 4. Penajaman menggunakan unsharp masking
    blur = cv2.GaussianBlur(kontras, (5, 5), 0)

    tajam = cv2.addWeighted(
        kontras, 1.5,
        blur, -0.5,
        0
    )

    # Kembalikan ke format BGR agar kompatibel
    # dengan fungsi visualisasi notebook
    hasil_akhir = cv2.cvtColor(
        tajam, cv2.COLOR_GRAY2BGR
    )

    return hasil_akhir

# Pengujian bentuk output tanpa menyimpan gambar berukuran besar ke dalam notebook.
uji = util.baca_citra(berkas[0])
hasil_uji = praproses(uji)
print('Ukuran asli:', uji.shape, '| setelah preprocessing:', hasil_uji.shape)

Ukuran asli: (1096, 827, 3) | setelah preprocessing: (1024, 773, 3)


**Alasan langkah preprocessing:**

1. **Resize (maksimal 1024 piksel):** Menyesuaikan ukuran citra dengan membatasi sisi terpanjang menjadi 1024 piksel sambil mempertahankan rasio aspek. Tahap ini bertujuan mengurangi beban komputasi dan menyeragamkan batas ukuran citra tanpa mengubah proporsi objek.

2. **Grayscale:** Mengonversi citra BGR menjadi citra keabuan menggunakan `cv2.COLOR_BGR2GRAY`. Tahap ini menyederhanakan informasi warna menjadi intensitas piksel sehingga proses peningkatan kontras dan analisis tepi lebih mudah dilakukan.

3. **CLAHE (Contrast Limited Adaptive Histogram Equalization):** Meningkatkan kontras lokal citra dengan parameter `clipLimit=2.0` dan `tileGridSize=(8,8)`. CLAHE digunakan untuk membantu memperjelas detail pelat dan karakter pada area yang memiliki pencahayaan atau kontras kurang merata, sekaligus membatasi penguatan kontras berlebihan.

4. **Sharpening (Unsharp Masking):** Mempertajam detail citra menggunakan Gaussian Blur berukuran kernel `(5,5)` dan `cv2.addWeighted()` dengan bobot `1.5` untuk citra asli serta `-0.5` untuk citra blur. Tahap ini bertujuan mempertegas tepi karakter dan batas pelat. Penajaman juga berpotensi memperkuat noise, sehingga kualitas hasil perlu diamati.

5. **Konversi kembali ke BGR:** Mengubah citra hasil preprocessing dari satu kanal grayscale menjadi tiga kanal BGR agar sesuai dengan format citra yang digunakan oleh fungsi visualisasi dan metode pengolahan pada repository.

**Hasil pengujian awal:** Citra `motor_001.jpeg` berukuran 4000 × 3000 piksel berhasil diproses menjadi 1024 × 768 piksel. Hasil visual menunjukkan peningkatan ketegasan detail karakter pelat, tetapi efektivitas preprocessing untuk keseluruhan dataset masih perlu diuji menggunakan citra dengan kondisi pencahayaan, sudut, dan kualitas yang berbeda.

## 4. Gambar sebelum–sesudah

Sel ini menyimpan 12 contoh ke folder `figures/` di Drive. Unggah minimal 10 ke `outputs/figures/` di repo (lihat `PANDUAN.md`).

Gambar versi publik yang sudah disensor dikelola terpisah; dataset utama dan gambar orisinal untuk evaluasi tidak diubah.

In [6]:
FOLDER_FIGUR.mkdir(parents=True, exist_ok=True)
diproses = 0
for p in berkas[:12]:
    asli = util.ubah_ukuran_maks(util.baca_citra(p), 1024)
    util.simpan_sebelum_sesudah(asli, praproses(asli), FOLDER_FIGUR / f"pra_{p.stem}.png")
    diproses += 1
print(diproses, 'gambar baru/sudah diperbarui di', FOLDER_FIGUR)

12 gambar baru/sudah diperbarui di /content/drive/MyDrive/pcd2026-ti23ka-t1/figures


In [7]:

from pathlib import Path

REPO = Path("/content/repo")

anggota = """kelas: ti23ka
topik: t1
anggota:
  - github: nuryansyahsw27
    nama: Nuryansyah
    peran: data
  - github: riyadhsmit1
    nama: Riyadh
    peran: klasik
  - github: kautsarapp
    nama: Kautsar
    peran: modern
  - github: Oscarmulya
    nama: Oscar
    peran: evaluasi
"""

(REPO / "ANGGOTA.yml").write_text(
    anggota, encoding="utf-8"
)

print("ANGGOTA.yml berhasil diperbarui.")
print(anggota)


ANGGOTA.yml berhasil diperbarui.
kelas: ti23ka
topik: t1
anggota:
  - github: nuryansyahsw27
    nama: Nuryansyah
    peran: data
  - github: riyadhsmit1
    nama: Riyadh
    peran: klasik
  - github: kautsarapp
    nama: Kautsar
    peran: modern
  - github: Oscarmulya
    nama: Oscar
    peran: evaluasi



## Ringkasan Tahap 1

- Dataset disimpan di Google Drive kelompok (bukan di GitHub).
- Statistik dan anotasi dihitung dari file aktif saat runtime dijalankan.
- Visualisasi sebelum–sesudah dibuat dengan satu fungsi `praproses`.
- Minimal 10 visualisasi tersensor dan 5 contoh dataset tersensor dimasukkan ke repository.
- Karakter pelat dan kotak harus diverifikasi sebelum dinilai sebagai anotasi final.

**Simpan hasil:** `Runtime → Restart session and run all`, pastikan seluruh sel berhasil, kemudian simpan salinan ke path resmi `notebooks/01_dataset_preprocessing.ipynb` pada branch pribadi, bukan `main`.